# 4회차 실습: 선형시스템을 **눈으로**, 그리고 AI에서 어디에 쓰이나

1. **해의 세 경우를 그림으로** — 가우스 소거가 말하는 유일·없음·무수가 기하적으로 무엇인가 (Row picture)
2. **AI 응용** — 모델 학습 = $A\mathbf{x}=\mathbf{b}$ 풀기, 그리고 특징 중복 = '무수한 해' = **다중공선성 (multicollinearity)**

In [13]:
%pip install -q koreanize-matplotlib
import numpy as np, matplotlib.pyplot as plt, koreanize_matplotlib  # noqa: F401
CR, NV, GD = '#862633', '#22386B', '#9A6B00'
rng = np.random.default_rng(0)


Note: you may need to restart the kernel to use updated packages.


## ① 해의 세 경우 = 두 직선의 관계 (Row picture)

$2$변수 방정식 하나는 평면 위 **직선** 하나다. 두 방정식(=두 직선)의 위치 관계가 곧 해의 개수다.

- **만난다** → 교점 하나 = **유일해**  ·  **평행** → 만날 수 없음 = **해 없음**  ·  **겹친다** → 직선 전체 = **무수한 해**

가우스 소거의 결과(pivot 패턴)가 이 세 그림 중 어느 것인지 알려주는 것이다.

**● 유일해 (교차)**

$$
A=\begin{bmatrix}1 & 1\\ 1 & -1\end{bmatrix},\quad
\mathbf{b}=\begin{bmatrix}3\\ 1\end{bmatrix},\quad
[A\mid\mathbf{b}]=\left[\begin{array}{cc|c}1 & 1 & 3\\ 1 & -1 & 1\end{array}\right],\quad
\begin{cases}x+y=3\\ x-y=1\end{cases}
$$

**● 해 없음 (평행)**

$$
A=\begin{bmatrix}1 & 1\\ 1 & 1\end{bmatrix},\quad
\mathbf{b}=\begin{bmatrix}1\\ 3\end{bmatrix},\quad
[A\mid\mathbf{b}]=\left[\begin{array}{cc|c}1 & 1 & 1\\ 1 & 1 & 3\end{array}\right],\quad
\begin{cases}x+y=1\\ x+y=3\end{cases}
$$

**● 무수한 해 (겹침)**

$$
A=\begin{bmatrix}1 & 1\\ 2 & 2\end{bmatrix},\quad
\mathbf{b}=\begin{bmatrix}2\\ 4\end{bmatrix},\quad
[A\mid\mathbf{b}]=\left[\begin{array}{cc|c}1 & 1 & 2\\ 2 & 2 & 4\end{array}\right],\quad
\begin{cases}x+y=2\\ 2x+2y=4\end{cases}
$$

In [ ]:
cases = [('유일해: 두 직선이 만남', np.array([[1, 1], [1, -1]]), np.array([3, 1]), (2, 1)),
         ('해 없음: 평행 (안 만남)', np.array([[1, 1], [1, 1]]), np.array([1, 3]), None),
         ('무수한 해: 같은 직선',   np.array([[1, 1], [2, 2]]), np.array([2, 4]), None)]
xs = np.linspace(-1, 4, 100)

fig, ax = plt.subplots(1, 3, figsize=(11, 3.8))
for a, (ttl, A, b, sol) in zip(ax, cases):
    for (p, q), r, c, ls in zip(A, b, [CR, NV], ['-', '--']):
        a.plot(xs, (r - p*xs)/q, color=c, lw=2.6, ls=ls)        # px+qy=r 을 직선으로
    if sol:
        a.plot(*sol, 'o', color=GD, ms=11)
        a.annotate(f'해 {sol}', sol, (sol[0]+.2, sol[1]+.4), color=GD, fontweight='bold', fontsize=11)
    a.set_title(ttl, fontsize=12)
    a.set_aspect('equal'); a.grid(alpha=.25)
    a.set_xlim(-1, 4); a.set_ylim(-1, 4)
plt.suptitle('두 방정식 = 두 직선   ·   교차/평행/겹침 = 유일해/해없음/무수한해', fontsize=13)
plt.tight_layout(); plt.show()

## ② AI 응용: 데이터를 지나는 곡선 찾기 = 선형시스템 풀기

3개의 점 $(0,1),(1,2),(2,5)$를 지나는 포물선 $y=a+bx+cx^2$의 계수 $(a,b,c)$를 구하는 것은, 점마다 식 하나씩 세운 **3×3 선형시스템**을 푸는 것이다. `np.linalg.solve`의 속이 이번 회차의 **가우스 소거(LU)**다.

$$\begin{cases} a = 1 \\ a+b+c=2 \\ a+2b+4c=5 \end{cases} \;\Leftrightarrow\; \begin{pmatrix}1&0&0\\1&1&1\\1&2&4\end{pmatrix}\begin{pmatrix}a\\b\\c\end{pmatrix}=\begin{pmatrix}1\\2\\5\end{pmatrix}$$

> 데이터 점이 미지수보다 **많으면** 모든 점을 지나는 곡선은 없어 **최소제곱**이 필요하다 (전치·역행렬을 배운 뒤 5회차·Part 2에서).


In [ ]:
A = np.array([[1., 0, 0], [1, 1, 1], [1, 2, 4]])   # 각 점의 (1, x, x^2)
b = np.array([1., 2, 5])                            # 각 점의 y
coef = np.linalg.solve(A, b)                        # 가우스소거로 3x3 풀기
print('포물선 계수 (a, b, c) =', np.round(coef, 3), '  → y = 1 + x^2')

xs = np.linspace(-0.3, 2.3, 100)
ys = coef[0] + coef[1]*xs + coef[2]*xs**2
plt.figure(figsize=(5.2, 3.6))
plt.scatter([0, 1, 2], [1, 2, 5], s=70, color=CR, zorder=5, label='주어진 3점')
plt.plot(xs, ys, color=NV, lw=2.2, label=f'맞춘 포물선  y = {coef[0]:.0f} + {coef[2]:.0f}x²')
plt.legend(fontsize=10); plt.title('3점을 지나는 포물선 = 3×3 선형시스템의 해', fontsize=12)
plt.xlabel('x'); plt.ylabel('y'); plt.grid(alpha=.25); plt.tight_layout(); plt.show()


## ③ 특징이 **중복**되면? — 4회차 '무수한 해'가 데이터에서

한 특징이 다른 특징의 배수이면($x_3 = 2x_1$, 새 정보 없음) $A\mathbf{x}=\mathbf{b}$에 **free variable**이 생겨 해가 하나로 안 정해진다 — 4회차의 **'무수한 해'** 그대로다. ML에서는 이 현상을 **다중공선성(multicollinearity)**이라 부른다 (정식은 Part 2 회귀).


In [16]:
X2 = np.c_[np.ones_like(x), x, 2*x]           # 3열 = 2×(2열): 중복 특징
wa = np.array([1., 2., 0.]);  wb = np.array([1., 0., 1.])   # 서로 다른 두 해
print('두 w가 같은 예측을 주는가?', np.allclose(X2 @ wa, X2 @ wb), ' (둘 다 y = 1 + 2x)')
print('→ 해가 하나로 안 정해짐 = 무수한 해 (free variable)')


rank(X2) = 2 / 열 3개  → 랭크 부족
det(XᵀX) = 0.0  (≈0 → 특이, 유일해 없음)
두 해의 예측이 같은가? True  (둘 다 y = 1 + 2x)


In [ ]:
# 중복 특징이면: 서로 다른 여러 w가 모두 같은 직선을 만든다 → 무수한 해
xs_fit = np.linspace(0, 5, 60)
X_fit = np.c_[np.ones_like(xs_fit), xs_fit, 2*xs_fit]

plt.figure(figsize=(5.8, 4.0))
plt.scatter(x, y, s=18, color=NV, alpha=.45, label='데이터')
for t, col in zip([0., 1., -2., 3.], [CR, GD, '#5B7C9A', '#4E7A4E']):
    w_t = np.array([1., 2. - 2*t, t])          # w=(1, 2-2t, t) → 항상 y=1+2x
    plt.plot(xs_fit, X_fit @ w_t, color=col, lw=1.8, alpha=.9,
             label=f'w=({w_t[0]:.0f}, {w_t[1]:.0f}, {w_t[2]:.0f})')
plt.title('서로 다른 해 → 같은 직선 (무수한 해)', fontsize=12)
plt.xlabel('x'); plt.ylabel('y'); plt.legend(fontsize=9, loc='upper left'); plt.grid(alpha=.25)
plt.tight_layout(); plt.show()


$x_3=2x_1$이라 $\mathbf{w}=(1,2,0)$이든 $(1,0,1)$이든 **같은 직선**이다 → ①의 '무수한 해'가 데이터에서 나타난 것. 새 정보 없는 중복 열은 해를 하나로 못 정한다.


## 정리

- **해의 세 경우**(4회차)는 곧 두 직선이 **교차/평행/겹침**. 가우스 소거의 pivot 패턴이 이 셋을 판정한다.
- **모델 학습 = $A\mathbf{x}=\mathbf{b}$ 풀기**. `solve`의 속은 가우스 소거.
- **독립 특징 → 유일해(안정)**, **중복 특징 → 무수한 해(불안정 = 다중공선성 · multicollinearity)**. 그래서 AI는 독립 특징을 쓰거나 **정규화**로 해를 하나로 고정한다.
